# Colab: attribute-based watermarking

GPU runtime recommended. Flow: clone → CPRF (Go) → PRC (Rust) → Python packages → Hugging Face login → `app.py` or benchmarks.

Put `HF_TOKEN=…` in `PROJECT_ROOT/.env` (optional) so login is non-interactive after §4 installs `python-dotenv`.

**Benchmark workflow (§9+):** each benchmark cell runs trials and writes JSON under `RESULTS_DIR/`. Plot cells call `benchmark_plot.py` on those files so you can re-chart without re-running generation.

**Watermark knobs (shared in §9, overridable per benchmark):**
- `PARTITION_MODE`: `static` (seeded unigram vocab halves) or `balanced` (per-step softmax-balanced masks from `randrecover`)
- `REDUNDANCY_LAYOUT`: `depth` (R interleaved codeword passes) or `block` (contiguous R-fold repeats)
- Detection reuses `wm.recover_channel_bits(..., generation_out=...)` once per transcript (required for balanced recovery)

Static partition unigram weights are packaged under `data/partition_weights/` (§6) and are only needed for `PARTITION_MODE="static"`.

After `git pull`, use **Runtime → Restart session** and re-run setup cells so Python picks up changed repo files.


In [ ]:
import sys

assert sys.version_info >= (3, 11), "Runtime → Change runtime type → Python 3.11+"

import torch

print("Python:", sys.version.split()[0], "| CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print()


## 1. Clone

Edit `REPO_OWNER` / `REPO_NAME` for a fork. Tree: `/content/<REPO_NAME>`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_OWNER = "maxraffel"
REPO_NAME = "attribute-based-watermarking"
PROJECT_ROOT = Path("/content") / REPO_NAME
url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not (PROJECT_ROOT / "watermarking.py").is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", url, str(PROJECT_ROOT)], check=True)
else:
    print("Repo present:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"
print("cwd:", PROJECT_ROOT.resolve())


## 2. Build CPRF (`cprf.so`)

Linux shared object. If `go` is missing or older than **1.24**, Go **1.24.4** is installed from [go.dev/dl](https://go.dev/dl) into `/usr/local/go` (requires `curl`).


In [ ]:
import os
import platform
import re
import shutil
import subprocess
from pathlib import Path


def _go_version_line() -> str | None:
    exe = shutil.which("go")
    if not exe:
        return None
    r = subprocess.run([exe, "version"], capture_output=True, text=True, timeout=60)
    if r.returncode != 0:
        return None
    return r.stdout


def _go_tuple_from_version(stdout: str) -> tuple[int, int, int] | None:
    m = re.search(r"go(\d+)\.(\d+)(?:\.(\d+))?", stdout)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)


def _go_ok() -> bool:
    line = _go_version_line()
    if not line:
        return False
    t = _go_tuple_from_version(line)
    return t is not None and t >= (1, 24, 0)


def _prepend_path(bin_dir: str) -> None:
    os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")


def _ensure_go(timeout_s: int = 420) -> str:
    if _go_ok():
        g = shutil.which("go")
        if not g:
            raise RuntimeError("go in PATH reported ok but shutil.which failed")
        return g

    local = Path("/usr/local/go/bin/go")
    if local.is_file():
        _prepend_path(str(local.parent))
        if _go_ok():
            return str(local)

    if not shutil.which("curl"):
        raise RuntimeError("curl is required to install Go on this VM")

    GO_VER = "1.24.4"
    arch = {
        "x86_64": "amd64",
        "AMD64": "amd64",
        "aarch64": "arm64",
        "arm64": "arm64",
    }.get(platform.machine(), "amd64")
    tgz = Path(f"/tmp/go{GO_VER}.linux-{arch}.tar.gz")
    url = f"https://go.dev/dl/go{GO_VER}.linux-{arch}.tar.gz"
    print("Installing Go", GO_VER, "from", url)
    subprocess.run(
        [
            "curl",
            "-fL",
            "--retry",
            "3",
            "--connect-timeout",
            "30",
            "-o",
            str(tgz),
            url,
        ],
        check=True,
        stdin=subprocess.DEVNULL,
        timeout=timeout_s,
    )
    if not tgz.is_file() or tgz.stat().st_size < 1024 * 1024:
        raise RuntimeError("Go download failed or file too small")

    goroot = Path("/usr/local/go")
    if goroot.is_dir():
        shutil.rmtree(goroot)
    subprocess.run(
        ["tar", "-C", "/usr/local", "-xzf", str(tgz)],
        check=True,
        stdin=subprocess.DEVNULL,
        timeout=timeout_s,
    )
    _prepend_path("/usr/local/go/bin")
    g = "/usr/local/go/bin/go"
    subprocess.run([g, "version"], check=True, timeout=60)
    return g


cprf_dir = PROJECT_ROOT / "cprf"
so_path = cprf_dir / "cprf.so"
go_exe = _ensure_go()

if not so_path.is_file():
    subprocess.run(
        [go_exe, "build", "-o", "cprf.so", "-buildmode=c-shared", "cprf.go"],
        cwd=cprf_dir,
        check=True,
    )
assert so_path.is_file(), "cprf.so missing"
print("CPRF:", so_path)


## 3. Build PRC (Rust + maturin)

Installs Rust via rustup if needed, then `maturin build --release` and `pip install` the newest wheel under `prc/target/wheels/`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

cargo_bin = Path.home() / ".cargo" / "bin"
if not (cargo_bin / "rustc").exists():
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        shell=True,
        check=True,
    )
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)

cp = subprocess.run(
    ["maturin", "build", "--release", "-m", "prc/Cargo.toml"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if cp.returncode != 0:
    if cp.stdout:
        print(cp.stdout)
    if cp.stderr:
        print(cp.stderr, file=sys.stderr)
    cp.check_returncode()

wheel_dir = PROJECT_ROOT / "prc" / "target" / "wheels"
wheels = sorted(wheel_dir.glob("*.whl"), key=lambda p: p.stat().st_mtime, reverse=True)
if not wheels:
    raise FileNotFoundError("No wheel in prc/target/wheels after maturin build")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])], check=True)
print("PRC:", wheels[0].name)


## 4. Python dependencies

Colab ships PyTorch with CUDA. `sympy` is reinstalled first (avoids a known Colab/transformers import clash). Adds packages used by optional chart/matrix cells.


In [ ]:
%pip install -q --upgrade --force-reinstall "sympy>=1.13,<2"
# rich<14: Colab's bigframes/pyiceberg pin rich below 14/15; we only need Console/Table/Progress.
%pip install -q "transformers==5.5.4" "rich>=12.4.4,<14" "keybert>=0.9" sentencepiece accelerate python-dotenv matplotlib pandas


## 5. Hugging Face login

Accept the **Llama 3.2** license on the model card. With `HF_TOKEN` / `HUGGING_FACE_HUB_TOKEN` in the environment or in `PROJECT_ROOT/.env`, login is automatic; otherwise this cell runs `notebook_login()`.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login, notebook_login

_env = PROJECT_ROOT / ".env"
if _env.is_file():
    load_dotenv(_env)

_token = os.environ.get("HF_TOKEN", "").strip() or os.environ.get(
    "HUGGING_FACE_HUB_TOKEN", ""
).strip()
if _token:
    login(token=_token, add_to_git_credential=False)
    print("HF: logged in from token.")
else:
    notebook_login()


## 6. Partition unigram weights (packaged)

Used only when `PARTITION_MODE="static"`. Watermark masks load **static** token-frequency weights from `data/partition_weights/`. Generation and detection **only load** that file — they do not recount a corpus.

`PARTITION_MODE="balanced"` builds masks from each step's post-processor probabilities and does **not** need this artifact.

After clone, confirm the artifact for `WATERMARK_LLM_ID` is present (static mode). Rebuild **only** when changing the tokenizer / vocab (then commit the new `.pt` into the repo):

```sh
uv sync --extra weights   # or: pip install datasets
python scripts/build_partition_weights.py --model-id meta-llama/Llama-3.2-1B-Instruct
```


In [ ]:
import os
from pathlib import Path

os.chdir(PROJECT_ROOT)

import randrecover as rr
from transformers import AutoTokenizer

_llm = WATERMARK_LLM_ID if "WATERMARK_LLM_ID" in globals() else "meta-llama/Llama-3.2-1B-Instruct"
_tok = AutoTokenizer.from_pretrained(_llm, token=True)
_v = max(int(getattr(_tok, "vocab_size", 0) or 0), len(_tok))
_key = rr.partition_weights_key(_tok, _v, tokenizer_id=_llm)
_path = rr.packaged_partition_weights_path(_key)
print("expected:", _path)
if not _path.is_file():
    raise FileNotFoundError(
        f"Missing packaged partition weights at {_path}. "
        "Run scripts/build_partition_weights.py once and commit the .pt, "
        "or pull the latest repo data/partition_weights/."
    )
_w, _meta = rr.load_partition_weights_file(_path)
rr.configure_partition_weights(_w, key=_key)
print(
    "loaded weights:",
    f"numel={_w.numel()}",
    f"corpus={_meta.get('corpus')}",
    f"tokens_counted={_meta.get('num_tokens_counted')}",
)


## 7. `git pull`

Optional. Re-run §2–§3 only if native code changed.


In [ ]:
import subprocess

if not (PROJECT_ROOT / ".git").is_dir():
    raise FileNotFoundError("Run §1 first.")

subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
log = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--oneline"],
    capture_output=True,
    text=True,
    check=True,
)
print(log.stdout.strip())


## 8. Demo (`app.py`)

Edit model id, sampling, inference dtype (`fp32` / `fp16` / `bf16`), and protocol constants in the next cell (`CODE_LENGTH`, `WM_BIT_REDUNDANCY`, `PARTITION_MODE`, `REDUNDANCY_LAYOUT`). `SystemExit` from failed checks is caught so Jupyter does not print a traceback for a nonzero exit code. Changing dtype (or model id) unloads weights so the next run reloads with the new setting.


In [ ]:
import os

os.chdir(PROJECT_ROOT)

WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 0  # 0 = disabled (HF convention)
INFERENCE_DTYPE = "fp16"  # "fp32" | "fp16" | "bf16"

import app as app_mod
import model

model.configure(
    model_id=WATERMARK_LLM_ID,
    temperature=float(TEMPERATURE),
    top_p=float(TOP_P),
    top_k=int(TOP_K),
    inference_dtype=INFERENCE_DTYPE,
)
app_mod.CODE_LENGTH = 100
app_mod.WM_BIT_REDUNDANCY = 7
app_mod.MODULUS = 1024
# ``static`` | ``balanced``; ``depth`` | ``block``
app_mod.PARTITION_MODE = "balanced"
app_mod.REDUNDANCY_LAYOUT = "depth"

try:
    app_mod.main()
except SystemExit as exc:
    if exc.code not in (0, None):
        print("app.py exit:", repr(exc.code), "(0 = all checks passed)")


## 9. Shared benchmark settings

Edit **LLM / sampling / inference dtype / default prompts / watermark modes** here. Each later benchmark cell has its own protocol knobs (code length, redundancy, runs, sweeps, ...) and inherits `BENCHMARK_PARTITION_MODE` / `BENCHMARK_REDUNDANCY_LAYOUT` unless overridden. Results land in `RESULTS_DIR/` as JSON for `benchmark_plot.py`.

Benchmark summaries print as **plain ASCII tables**. Rich is only used for the progress bar during runs.


In [ ]:
import os
import sys
from pathlib import Path

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

RESULTS_DIR = root / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- shared LLM / sampling / dtype (edit here) ---
BENCHMARK_LLM_MODEL: str | None = None  # None -> WATERMARK_LLM_ID from section 1/7
BENCHMARK_TEMPERATURE = 1.0
BENCHMARK_TOP_P = 0.95
BENCHMARK_TOP_K = 0  # 0 = disabled (HF convention)
BENCHMARK_INFERENCE_DTYPE = "fp16"  # "fp32" | "fp16" | "bf16"

# Watermark channel / partition (edit here; per-benchmark cells can override)
BENCHMARK_PARTITION_MODE = "balanced"  # "static" | "balanced"
BENCHMARK_REDUNDANCY_LAYOUT = "depth"  # "depth" | "block"

# Default prompt cases for benchmarks that don't override; "id:prompt"; empty -> COMPREHENSIVE_PROMPT_CASES
BENCHMARK_PROMPT_CASES: list[str] = [
    "medicine_stem_cell:Explain how stem cell therapy is being used in regenerative medicine.",
    "economics_min_wage:Explain the economic effects of raising the minimum wage on employment and businesses.",
    "art_surrealism:Explain how Surrealist artists used dream imagery to challenge reality and logic.",
    "software_breakthroughs:Break down the most influential software breakthroughs in history.",
    "sports_strategy_performance:Explain the role of strategy and teamwork in achieving success in sports.",
]

import model
import watermarking as wm
from benchmark_policy_detection import COMPREHENSIVE_PROMPT_CASES, parse_prompt_case

llm = BENCHMARK_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
model.configure(
    model_id=str(llm).strip() if llm and str(llm).strip() else None,
    temperature=float(BENCHMARK_TEMPERATURE),
    top_p=float(BENCHMARK_TOP_P),
    top_k=int(BENCHMARK_TOP_K),
    inference_dtype=BENCHMARK_INFERENCE_DTYPE,
)
wm.set_partition_mode(BENCHMARK_PARTITION_MODE)
wm.set_redundancy_layout(BENCHMARK_REDUNDANCY_LAYOUT)

benchmark_cases = (
    [parse_prompt_case(s) for s in BENCHMARK_PROMPT_CASES]
    if BENCHMARK_PROMPT_CASES
    else list(COMPREHENSIVE_PROMPT_CASES)
)

print("results:", RESULTS_DIR.resolve())
print("LLM:", model.MODEL_ID)
print(
    "sampling:",
    f"temperature={model.SAMPLING['temperature']}",
    f"top_p={model.SAMPLING['top_p']}",
    f"top_k={model.SAMPLING['top_k']}",
)
print("inference_dtype:", model.inference_dtype_label())
print("partition_mode:", wm.PARTITION_MODE)
print("redundancy_layout:", wm.REDUNDANCY_LAYOUT)
print("prompt cases:", [sid for sid, _ in benchmark_cases])


## 10. Policy detection benchmark

Runs the full `app.py`-shaped protocol over `benchmark_cases` (balanced recovery uses `generation_out`). Edit protocol knobs in the next cell. Writes `results/benchmark_policy_detection.json`.


In [ ]:
import benchmark_io
from benchmark_policy_detection import make_benchmark_console, run_benchmark_with_summary

# --- this benchmark only ---
POLICY_MODULUS = 1024
POLICY_CODE_LENGTH = 100
POLICY_WM_BIT_REDUNDANCY = 7
POLICY_PARTITION_MODE = BENCHMARK_PARTITION_MODE
POLICY_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
POLICY_RUNS = 3
POLICY_REUSE_KEY = False  # False => fresh CPRF master key every trial

policy_json = RESULTS_DIR / "benchmark_policy_detection.json"
exit_code, summary = run_benchmark_with_summary(
    prompt_cases=benchmark_cases,
    runs=int(POLICY_RUNS),
    modulus=int(POLICY_MODULUS),
    code_length=int(POLICY_CODE_LENGTH),
    fresh_key_per_trial=not POLICY_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    wm_bit_redundancy=int(POLICY_WM_BIT_REDUNDANCY),
    partition_mode=str(POLICY_PARTITION_MODE),
    redundancy_layout=str(POLICY_REDUNDANCY_LAYOUT),
    quiet=False,
)
benchmark_io.save_policy_summary(
    policy_json,
    summary=summary,
    exit_code=exit_code,
    runs=int(POLICY_RUNS),
    fresh_key_per_trial=not POLICY_REUSE_KEY,
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
)
print("exit:", exit_code, "| wrote", policy_json)


## 11. Sweep: FPR vs code length

One policy benchmark per entry in `FPR_SWEEP_CODE_LENGTHS`, plus a PRC random baseline. Edit sweep knobs in the next cell. Writes `results/benchmark_fpr_vs_code_length.json`.


In [ ]:
import random
from benchmark_policy_detection import (
    make_benchmark_console,
    run_fpr_vs_code_length_sweep,
    save_fpr_sweep_results,
)

# --- this benchmark only ---
FPR_SWEEP_CODE_LENGTHS = [128, 256, 384]
FPR_SWEEP_PRC_MONTE_CARLO = 100_000
FPR_MODULUS = 1024
FPR_WM_BIT_REDUNDANCY = 7
FPR_PARTITION_MODE = BENCHMARK_PARTITION_MODE
FPR_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
FPR_RUNS = 3
FPR_REUSE_KEY = False

fpr_json = RESULTS_DIR / "benchmark_fpr_vs_code_length.json"
lengths, metrics, exit_codes = run_fpr_vs_code_length_sweep(
    prompt_cases=benchmark_cases,
    code_lengths=FPR_SWEEP_CODE_LENGTHS,
    runs=int(FPR_RUNS),
    modulus=int(FPR_MODULUS),
    wm_bit_redundancy=int(FPR_WM_BIT_REDUNDANCY),
    fresh_key_per_trial=not FPR_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    partition_mode=str(FPR_PARTITION_MODE),
    redundancy_layout=str(FPR_REDUNDANCY_LAYOUT),
    prc_monte_carlo_trials=int(FPR_SWEEP_PRC_MONTE_CARLO),
    rng=random.Random(42),
    quiet=True,
)
save_fpr_sweep_results(
    fpr_json,
    code_lengths=lengths,
    metrics=metrics,
    exit_codes=exit_codes,
    runs=int(FPR_RUNS),
    modulus=int(FPR_MODULUS),
    wm_bit_redundancy=int(FPR_WM_BIT_REDUNDANCY),
    prc_monte_carlo_trials=int(FPR_SWEEP_PRC_MONTE_CARLO),
    prompt_cases=benchmark_cases,
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
)
for L, ex, fa in zip(lengths, exit_codes, metrics["scheme_fpr_all_runs"]):
    print(f"code_length={L} exit={ex} FPR_all={fa:.6g}")
print("wrote", fpr_json)


## 12. Sweep: TPR vs WM bit redundancy

Fixed logical `TPR_CODE_LENGTH`; sweeps `TPR_WM_BIT_REDUNDANCY_VALUES` at the shared partition/layout. Edit knobs in the next cell. Writes `results/benchmark_tpr_vs_wm_bit_redundancy.json`.


In [ ]:
from benchmark_policy_detection import (
    make_benchmark_console,
    run_tpr_vs_wm_bit_redundancy_sweep,
    save_tpr_sweep_results,
)

# --- this benchmark only ---
TPR_WM_BIT_REDUNDANCY_VALUES = [1, 3, 5, 7]
TPR_CODE_LENGTH = 100
TPR_MODULUS = 1024
TPR_PARTITION_MODE = BENCHMARK_PARTITION_MODE
TPR_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
TPR_RUNS = 3
TPR_REUSE_KEY = False

tpr_json = RESULTS_DIR / "benchmark_tpr_vs_wm_bit_redundancy.json"
redundancies, metrics, exit_codes = run_tpr_vs_wm_bit_redundancy_sweep(
    prompt_cases=benchmark_cases,
    wm_bit_redundancy_values=TPR_WM_BIT_REDUNDANCY_VALUES,
    code_length=int(TPR_CODE_LENGTH),
    runs=int(TPR_RUNS),
    modulus=int(TPR_MODULUS),
    fresh_key_per_trial=not TPR_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    partition_mode=str(TPR_PARTITION_MODE),
    redundancy_layout=str(TPR_REDUNDANCY_LAYOUT),
    quiet=True,
)
save_tpr_sweep_results(
    tpr_json,
    wm_bit_redundancy_values=redundancies,
    metrics=metrics,
    exit_codes=exit_codes,
    code_length=int(TPR_CODE_LENGTH),
    runs=int(TPR_RUNS),
    modulus=int(TPR_MODULUS),
    prompt_cases=benchmark_cases,
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
)
for R, ex, ta in zip(redundancies, exit_codes, metrics["tpr_all_runs"]):
    print(f"wm_bit_redundancy={R} exit={ex} TPR_all={ta:.6g}")
print("wrote", tpr_json)


## 13. Detection matrices

Label-conditioned (`|V|x|V|`) and prompt-conditioned (`|V|x|P|`) detection rates. Edit knobs and prompt-matrix cases in the next cell. JSON only; plot in section 14.


In [ ]:
import pandas as pd
import benchmark_io
from benchmark_policy_detection import (
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark_label_conditioned_matrix,
    run_benchmark_prompt_conditioned_matrix,
)

# --- this benchmark only ---
MATRIX_MODULUS = 1024
MATRIX_CODE_LENGTH = 100
MATRIX_WM_BIT_REDUNDANCY = 7
MATRIX_PARTITION_MODE = BENCHMARK_PARTITION_MODE
MATRIX_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
MATRIX_RUNS = 3
MATRIX_REUSE_KEY = False
PROMPT_MATRIX_CASES: list[str] = [
    "sports_econ:Explain the economic nuance and impact of Drake Maye during his college football career at North Carolina.",
    "art_software:Explain how software has transformed the art world.",
    "medicine_software:Explain how software has transformed the practice of medicine.",
]

prompt_matrix_cases = (
    [parse_prompt_case(s) for s in PROMPT_MATRIX_CASES]
    if PROMPT_MATRIX_CASES
    else list(benchmark_cases)
)

label_json = RESULTS_DIR / "benchmark_label_conditioned_detection_matrix.json"
ex_label, label_mx = run_benchmark_label_conditioned_matrix(
    prompt_cases=benchmark_cases,
    runs=int(MATRIX_RUNS),
    modulus=int(MATRIX_MODULUS),
    code_length=int(MATRIX_CODE_LENGTH),
    fresh_key_per_trial=not MATRIX_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    wm_bit_redundancy=int(MATRIX_WM_BIT_REDUNDANCY),
    partition_mode=str(MATRIX_PARTITION_MODE),
    redundancy_layout=str(MATRIX_REDUNDANCY_LAYOUT),
    quiet=True,
)
benchmark_io.save_label_matrix(
    label_json,
    matrix=label_mx,
    exit_code=ex_label,
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
)

prompt_json = RESULTS_DIR / "benchmark_prompt_conditioned_detection_matrix.json"
ex_prompt, prompt_mx = run_benchmark_prompt_conditioned_matrix(
    prompt_cases=prompt_matrix_cases,
    runs=int(MATRIX_RUNS),
    modulus=int(MATRIX_MODULUS),
    code_length=int(MATRIX_CODE_LENGTH),
    fresh_key_per_trial=not MATRIX_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    wm_bit_redundancy=int(MATRIX_WM_BIT_REDUNDANCY),
    partition_mode=str(MATRIX_PARTITION_MODE),
    redundancy_layout=str(MATRIX_REDUNDANCY_LAYOUT),
    quiet=True,
)
benchmark_io.save_prompt_matrix(
    prompt_json,
    matrix=prompt_mx,
    exit_code=ex_prompt,
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
)

vocab = list(label_mx.vocab)
cols = list(prompt_mx.column_prompt_ids)
for name, mx_obj, ex in (
    ("label", label_mx, ex_label),
    ("prompt", prompt_mx, ex_prompt),
):
    num_df = pd.DataFrame(mx_obj.numerators).T.loc[vocab, vocab if name == "label" else cols]
    den_df = pd.DataFrame(mx_obj.denominators).T.loc[vocab, vocab if name == "label" else cols]
    pct_df = (num_df / den_df.replace(0, pd.NA)) * 100.0
    print(f"{name} matrix exit={ex} |V|={len(vocab)}")
    display(pct_df.round(2))

print("wrote", label_json, "and", prompt_json)


## 14. BER diagnostics & watermark protocol benchmark

Decomposes BER by channel/recovery/PRC stage (`benchmark_ber_diagnostics.py`) and runs the batch protocol check (`benchmark_watermark.py`). Both honor `partition_mode` and `redundancy_layout`. Edit knobs in the next cell.


In [ ]:
from benchmark_ber_diagnostics import BerDiagConfig, run_ber_diagnostics, print_ber_diagnostics
from benchmark_watermark import BenchmarkConfig, run_benchmark, print_summary
from benchmark_io import make_benchmark_console
import benchmark_io

# --- this benchmark only ---
BER_MODULUS = 1024
BER_CODE_LENGTH = 100
BER_WM_BIT_REDUNDANCY = 7
BER_PARTITION_MODE = BENCHMARK_PARTITION_MODE
BER_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
BER_PROMPT_LIMIT = 3  # first N shared prompt cases

WM_MODULUS = 1024
WM_CODE_LENGTH = 100
WM_BIT_REDUNDANCY = 7
WM_PARTITION_MODE = BENCHMARK_PARTITION_MODE
WM_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
WM_RUNS = 3
WM_REUSE_BASELINE = True

bench_console = make_benchmark_console()
llm_id = str(llm).strip() if llm and str(llm).strip() else None

ber_json = RESULTS_DIR / "benchmark_ber_diagnostics.json"
ber_prompts = [prompt for _, prompt in benchmark_cases[: int(BER_PROMPT_LIMIT)]]
ber_cfg = BerDiagConfig(
    modulus=int(BER_MODULUS),
    code_length=int(BER_CODE_LENGTH),
    wm_bit_redundancy=int(BER_WM_BIT_REDUNDANCY),
    redundancy_layout=str(BER_REDUNDANCY_LAYOUT),
    partition_mode=str(BER_PARTITION_MODE),
    model_id=llm_id,
)
ber_summary = run_ber_diagnostics(ber_prompts, ber_cfg)
print_ber_diagnostics(ber_summary, console=bench_console)
benchmark_io.save_ber_diagnostics(
    ber_json,
    summary=ber_summary,
    llm_model_id=ber_cfg.model_id,
)
print("wrote", ber_json)

wm_json = RESULTS_DIR / "benchmark_watermark_protocol.json"
wm_cfg = BenchmarkConfig(
    modulus=int(WM_MODULUS),
    code_length=int(WM_CODE_LENGTH),
    wm_bit_redundancy=int(WM_BIT_REDUNDANCY),
    redundancy_layout=str(WM_REDUNDANCY_LAYOUT),
    partition_mode=str(WM_PARTITION_MODE),
    model_id=llm_id,
    repeats_per_prompt=int(WM_RUNS),
    reuse_baseline=bool(WM_REUSE_BASELINE),
)
wm_prompts = [prompt for _, prompt in benchmark_cases]
wm_runs = run_benchmark(wm_prompts, wm_cfg)
print_summary(wm_prompts, wm_runs, wm_cfg, console=bench_console)
benchmark_io.save_watermark_benchmark(
    wm_json,
    config=wm_cfg,
    prompts=wm_prompts,
    runs=wm_runs,
    llm_model_id=wm_cfg.model_id,
)
print("wrote", wm_json)


## 15. Plot from saved JSON

Uses `benchmark_plot.py` — re-run this cell anytime without re-running generation. Set `PLOT_WITH_CI = True` for Wilson intervals on FPR/TPR sweeps.


In [ ]:
from benchmark_plot import plot_from_file

PLOT_WITH_CI = True

plot_specs = [
    ("fpr", RESULTS_DIR / "benchmark_fpr_vs_code_length.json", {"with_ci": PLOT_WITH_CI}),
    ("tpr", RESULTS_DIR / "benchmark_tpr_vs_wm_bit_redundancy.json", {"with_ci": PLOT_WITH_CI}),
    ("label-matrix", RESULTS_DIR / "benchmark_label_conditioned_detection_matrix.json", {}),
    ("prompt-matrix", RESULTS_DIR / "benchmark_prompt_conditioned_detection_matrix.json", {}),
    (
        "prompt-matrix",
        RESULTS_DIR / "benchmark_prompt_conditioned_detection_matrix.json",
        {"xmatch": True},
    ),
    ("ber", RESULTS_DIR / "benchmark_ber_diagnostics.json", {}),
]

for kind, path, kwargs in plot_specs:
    if not path.is_file():
        print("skip (missing):", path.name, f"[{kind}]")
        continue
    out = plot_from_file(kind, path, show=True, **kwargs)
    print(kind, "->", out)

## 16. Custom prompt (`app.main()`)

Reloads `app`, patches prompt / lengths / partition / layout / sampling / inference dtype, runs `main()` without editing `app.py`.


In [ ]:
import importlib
import app as app_mod
import model

APP_CUSTOM_PROMPT = "Explain how advances in battery chemistry changed electric vehicle adoption."
APP_CUSTOM_CODE_LENGTH = 100
APP_CUSTOM_WM_BIT_REDUNDANCY = 7
APP_CUSTOM_MODULUS = 1024
APP_CUSTOM_PARTITION_MODE = BENCHMARK_PARTITION_MODE
APP_CUSTOM_REDUNDANCY_LAYOUT = BENCHMARK_REDUNDANCY_LAYOUT
APP_CUSTOM_TEMPERATURE = 1.0
APP_CUSTOM_TOP_P = 0.95
APP_CUSTOM_TOP_K = 0
APP_CUSTOM_INFERENCE_DTYPE = "fp16"  # "fp32" | "fp16" | "bf16"

app_mod = importlib.reload(app_mod)
app_mod.PROMPT = str(APP_CUSTOM_PROMPT)
app_mod.CODE_LENGTH = int(APP_CUSTOM_CODE_LENGTH)
app_mod.WM_BIT_REDUNDANCY = int(APP_CUSTOM_WM_BIT_REDUNDANCY)
app_mod.MODULUS = int(APP_CUSTOM_MODULUS)
app_mod.PARTITION_MODE = str(APP_CUSTOM_PARTITION_MODE)
app_mod.REDUNDANCY_LAYOUT = str(APP_CUSTOM_REDUNDANCY_LAYOUT)
app_mod.MODEL_ID = str(llm).strip() if llm and str(llm).strip() else None
model.configure(
    model_id=app_mod.MODEL_ID,
    temperature=float(APP_CUSTOM_TEMPERATURE),
    top_p=float(APP_CUSTOM_TOP_P),
    top_k=int(APP_CUSTOM_TOP_K),
    inference_dtype=APP_CUSTOM_INFERENCE_DTYPE,
)

print(
    "app.main()",
    app_mod.PROMPT[:80],
    "...",
    f"partition={app_mod.PARTITION_MODE}",
    f"layout={app_mod.REDUNDANCY_LAYOUT}",
)
code = app_mod.main()
print("exit:", code, "(0 = all checks passed)")
